In [1]:
import numpy as np
import numpy.typing as npt
from astropy.time import Time
from sorts.propagator import SGP4
from sorts.space_object import SpaceObject
from sorts.radar.radars import get_radar
from sorts.types import Datetime64_us, Timedelta64_us, Float64_as_sec
from sorts.controller_v2.tracker_controller import TrackerController
from sorts.controller_v2.fence_scan_controller_new import FenceScanController
from sorts.schedule_v2 import Schedule, ExperimentDetail
from sorts.scheduler_v2.priority_scheduling import _priority_scheduling_df

# import for plottings
import pandas as pd
from sorts.plotting_deps import lp, lp_geo_data, geodatasets, gpd
from sorts import plots

The geodata is provided by © OpenStreetMap contributors and is made available here under the Open Database License (ODbL).


In [2]:
pd.set_option("display.expand_frame_repr", False)

In [3]:
epoch = Time(53005.0, format="mjd", scale="utc")  # 2004-01-01 00:00:00Z
# start_time = Time("2025-06-30 00:00:00")
# end_time = Time("2025-06-30 00:00:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
# start_time = Time("2025-01-01 04:04:00")
# end_time = Time("2025-01-01 04:04:01")
# control_slice_duration = np.timedelta64(10_000, "us")  # 10ms
start_time = Time("2025-01-01 02:45:00")
end_time = Time("2025-01-01 06:15:00")
control_slice_duration = np.timedelta64(int(60 * 1e6), "us")

eiscat3d = get_radar("eiscat3d", "stage1-array")

spobj = SpaceObject(
    SGP4,
    propagator_options={"settings": {"out_frame": "ITRF"}},
    a=7200e3,
    e=0.02,
    i=75,
    raan=86,
    aop=0,
    mu0=60,
    epoch=epoch,
    parameters={"d": 0.1},
)


exp_detail_0 = ExperimentDetail(
    id=0,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

exp_detail_1 = ExperimentDetail(
    id=1,
    coh_int_bandwidth=1.0,
    ipp=1.0,
    pulse_length=1.0,
    power=5000000.0,
    bandwidth=52.08333333333333,
    duty_cycle=1.0,
    noise_temp=150.0,
    slice_duration=control_slice_duration
)

time_arr: npt.NDArray[Datetime64_us] = np.arange(
    start_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    end_time.to_value("datetime64").astype("datetime64[us]"),  # type: ignore
    control_slice_duration,
)
# time_arr = time_arr[::4] # TODO: remove; strided to bring up the effects of scheduling
dt_arr: npt.NDArray[Timedelta64_us] = time_arr - epoch.to_value("datetime64").astype("datetime64[us]")  # type: ignore
dsec_arr: npt.NDArray[Float64_as_sec] = dt_arr.astype(np.float64) / 1e6  # type: ignore

ecefs = spobj.get_state(dsec_arr)

trackerController = TrackerController(
    tx_station=eiscat3d.tx[0],
    rx_stations=[],
    time=time_arr,
    space_object_states=ecefs,
    exp_detail=exp_detail_0,
    # azimuth_range=None,
    # elevation_range=None,
)

fenceScanController = FenceScanController(
    tx_station=eiscat3d.tx[0],
    rx_station=[],
    exp_datail=exp_detail_1,
    azimuth=90, # sweep from east to west
    min_elevation=30,
    pointings_per_cycle=40,
)

In [4]:
plots.ecef_states_positions_plot(ecefs)

In [5]:
tracker_schs = trackerController.generate()
tracker_tx_sch_df = tracker_schs.tx_schedule.as_dataframe()
tracker_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00
1,2025-01-01 02:46:00,17.521616,11.088799,0,2025-01-01 02:47:00
2,2025-01-01 02:47:00,17.125242,7.762307,0,2025-01-01 02:48:00
3,2025-01-01 02:48:00,16.748612,4.432886,0,2025-01-01 02:49:00
4,2025-01-01 02:49:00,16.387257,1.100487,0,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,62.053178,19.012611,0,2025-01-01 06:11:00
206,2025-01-01 06:11:00,62.507070,15.690203,0,2025-01-01 06:12:00
207,2025-01-01 06:12:00,62.958886,12.366929,0,2025-01-01 06:13:00
208,2025-01-01 06:13:00,63.410154,9.042348,0,2025-01-01 06:14:00


In [6]:
fence_schs = fenceScanController.generate(start_time, end_time)
fence_tx_sch_df = fence_schs.tx_schedule.as_dataframe()
fence_tx_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time
0,2025-01-01 02:45:00,90.0,30.000000,1,2025-01-01 02:46:00
1,2025-01-01 02:46:00,90.0,33.076923,1,2025-01-01 02:47:00
2,2025-01-01 02:47:00,90.0,36.153846,1,2025-01-01 02:48:00
3,2025-01-01 02:48:00,90.0,39.230769,1,2025-01-01 02:49:00
4,2025-01-01 02:49:00,90.0,42.307692,1,2025-01-01 02:50:00
...,...,...,...,...,...
205,2025-01-01 06:10:00,90.0,45.384615,1,2025-01-01 06:11:00
206,2025-01-01 06:11:00,90.0,48.461538,1,2025-01-01 06:12:00
207,2025-01-01 06:12:00,90.0,51.538462,1,2025-01-01 06:13:00
208,2025-01-01 06:13:00,90.0,54.615385,1,2025-01-01 06:14:00


In [13]:
# check schedule df memory usage
fence_tx_sch_df.memory_usage().sum()/1e6

np.float64(0.008528)

In [10]:
tx_sch = tracker_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [11]:
tx_sch = fence_schs.tx_schedule
plots.azel_polar_plot(tx_sch.pointing_az, tx_sch.pointing_el)

In [14]:
(tracker_schs.tx_schedule.meta, fence_schs.tx_schedule.meta)

({0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))},
 {1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(60000000,'us'))})

In [15]:
master_sch_meta = {0: exp_detail_0, 1: exp_detail_1}
master_sch_df = _priority_scheduling_df([tracker_schs.tx_schedule, fence_schs.tx_schedule])
master_sch_df

,stt_tstmp_us,pointing_az,pointing_el,exp_num,end_time,allowed_start_time,allowed_end_time,is_overlaped
0,2025-01-01 02:45:00,17.942607,14.412365,0,2025-01-01 02:46:00,2025-01-01 02:45:00,2025-01-01 02:46:00,False
1,2025-01-01 02:46:00,17.521616,11.088799,0,2025-01-01 02:47:00,2025-01-01 02:46:00,2025-01-01 02:47:00,False
2,2025-01-01 02:47:00,17.125242,7.762307,0,2025-01-01 02:48:00,2025-01-01 02:47:00,2025-01-01 02:48:00,False
3,2025-01-01 02:48:00,16.748612,4.432886,0,2025-01-01 02:49:00,2025-01-01 02:48:00,2025-01-01 02:49:00,False
4,2025-01-01 02:49:00,16.387257,1.100487,0,2025-01-01 02:50:00,2025-01-01 02:49:00,2025-01-01 02:50:00,False
...,...,...,...,...,...,...,...,...
205,2025-01-01 06:10:00,62.053178,19.012611,0,2025-01-01 06:11:00,2025-01-01 06:10:00,2025-01-01 06:11:00,False
206,2025-01-01 06:11:00,62.507070,15.690203,0,2025-01-01 06:12:00,2025-01-01 06:11:00,2025-01-01 06:12:00,False
207,2025-01-01 06:12:00,62.958886,12.366929,0,2025-01-01 06:13:00,2025-01-01 06:12:00,2025-01-01 06:13:00,False
208,2025-01-01 06:13:00,63.410154,9.042348,0,2025-01-01 06:14:00,2025-01-01 06:13:00,2025-01-01 06:14:00,False


In [10]:
master_sch = Schedule.from_dataframe(master_sch_df, master_sch_meta)
master_sch

Schedule(meta={0: ExperimentDetail(id=0, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us')), 1: ExperimentDetail(id=1, coh_int_bandwidth=1.0, ipp=1.0, pulse_length=1.0, power=5000000.0, bandwidth=52.08333333333333, duty_cycle=1.0, noise_temp=150.0, slice_duration=np.timedelta64(10000,'us'))}, stt_tstmp_us=array(['2025-06-30T00:00:00.000000', '2025-06-30T00:00:00.020000',
       '2025-06-30T00:00:00.040000', '2025-06-30T00:00:00.060000',
       '2025-06-30T00:00:00.080000', '2025-06-30T00:00:00.100000',
       '2025-06-30T00:00:00.120000', '2025-06-30T00:00:00.140000',
       '2025-06-30T00:00:00.160000', '2025-06-30T00:00:00.180000',
       '2025-06-30T00:00:00.200000', '2025-06-30T00:00:00.220000',
       '2025-06-30T00:00:00.240000', '2025-06-30T00:00:00.260000',
       '2025-06-30T00:00:00.280000', '2025-06-30T00:00:00.300000',
       '2025-06-30T00:00:00.320000'